# Principled Comparison of Robot Policy Performance

We illustrate the use of statistically rigorous sequential evaluation procedures to rapidly and reliably compare robot policy performance

### Import Lightweight Python Libraries

In [2]:
import numpy as np 
import os 
import sys
from tqdm import tqdm 

### Import Rigorous Policy Comparison Methods

In [16]:
# General tools for statistical hypothesis testing
from sequentialized_barnard_tests.base import Hypothesis, Decision

# Tools specifically for comparison under binary performance measures
from sequentialized_barnard_tests.step import StepTest
from sequentialized_barnard_tests.savi import SaviTest
from sequentialized_barnard_tests.batch import BarnardExactTest

#####
# New, general-purpose tools for arbitrary, bounded performance measures
#####
path_for_loading_nscore = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(path_for_loading_nscore)

# WSR METHOD: Unstructured nonparametric method (more general, but less sample efficient)
from nscore.wsr import WsrComparisonTest

# Theta-SAVI METHOD: Explicitly structured parametric method (less general, but more sample efficient)
from nscore.savi import PartialCreditSaviTest

# Our method (NSCORE): as general as WSR and as efficient as Theta-SAVI 
from nscore.nsm import BernoulliNsmTest
from nscore.nonparametric_nsm import ContinuousNsmTest

## The Setting of Policy Comparison

Recall (see [here](https://medium.com/toyotaresearch/statistical-thinking-for-robot-policy-evaluation-from-rigorous-a-b-testing-to-effective-0ae886fbd68d) for more details) that we are considering the problem of proving, at given confidence level $1 - \alpha$, that the mean performance of our robot policy is better than a baseline (i.e., the current state of the art). Because robot evaluation is expensive, we want to do this as quickly as possible. 

Mathematically, we will express the comparison in the following terms: 
-  $\mu_1 = \mu(\pi_1)$, the mean performance of the new policy
-  $\mu_0 = \mu(\pi_0)$, the mean performance of the state-of-the-art baseline

The question is precisely: determine if, given the available data, $\mu_1 > \mu_0 \text{ w.p.} \geq 1-\alpha$. 

However, the desiderata of our determination -- high confidence and small sample size -- are in conflict! Concluding that our new policy is better with high confidence requires us to be very careful in how we use the limited data at our disposal. 

Currently, the standard evaluation practice is to choose a batch size, $N$, run $N$ evaluations of each policy, and report the change in empirical performance. However, this can be very misleading, as the following example illustrates. 

### A Simple Mental Experiment: The "Evaluation Code Bug"

Imagine that you are pushing for a deadline. Though it'll be tight, you're almost there: all you have left to do is run the hardware evaluations of your new policy against the current state of the art baseline. All of the stuff is ready (after much suffering, of course), but in the last days of the crunch, everything goes without a hitch. 

There's just one problem -- a silent bug. Specifically: the variable loading the policy weights was inadvertantly hardcoded from an earlier debugging phase, and every time an evaluation was run, the baseline policy was loaded!

Imagine the actual data collection in that case: 
- For the `baseline data,' the true mean is $\mu_0$. 
- For the `new policy,' the true mean is ... also $\mu_0$!

What happens downstream? Well, let's imagine that the performance measure is binary (success / failure), and assume that 25 evaluations were run per policy. What is the chance of observing at least 8 percentage points of improvement empirically? 12 points? 16? 20?

I strongly encourage the interested reader to formulate a concrete estimate. For context: the correct answer will be somewhere between 0% and 50%; we will assume that the true baseline policy success rate is 57% (to choose a random prime number...). The answer will be printed at the end of the following code snippet. Additionally, note that the random seed can be freely adjusted to verify that the number is not itself a fluke result. 

In [10]:
n_fictitious_evaluation_sequences = 10000
n_evaluations_per_sequence = 25 

p0 = 0.57       # Baseline succeeds 57% of the time
p1 = 0.57       # Code bug means we were loading the baseline here as well! So it is also 57%

# Count the times we get at least 8 percentage point improvement from random noise. 
eight_pp_improvement = np.zeros(n_fictitious_evaluation_sequences)
twelve_pp_improvement = np.zeros(n_fictitious_evaluation_sequences)
sixteen_pp_improvement = np.zeros(n_fictitious_evaluation_sequences)
twenty_pp_improvement = np.zeros(n_fictitious_evaluation_sequences)

np.random.seed(42)

for i in range(n_fictitious_evaluation_sequences):
    data_baseline = np.random.binomial(1, p0, n_evaluations_per_sequence)
    data_new_policy = np.random.binomial(1, p1, n_evaluations_per_sequence)

    if np.mean(data_new_policy) >= (np.mean(data_baseline) + 0.08 - 1e-8):
        eight_pp_improvement[i] += 1.0 
    
    if np.mean(data_new_policy) >= (np.mean(data_baseline) + 0.12 - 1e-8):
        twelve_pp_improvement[i] += 1.0
    
    if np.mean(data_new_policy) >= (np.mean(data_baseline) + 0.16 - 1e-8):
        sixteen_pp_improvement[i] += 1.0
    
    if np.mean(data_new_policy) >= (np.mean(data_baseline) + 0.20 - 1e-8):
        twenty_pp_improvement[i] += 1.0


print(f"Percentage of cases where we observe 8 percentage point improvement from random noise: {100.*np.mean(eight_pp_improvement):0.2f}%")
print(f"Percentage of cases where we observe 12 percentage point improvement from random noise: {100.*np.mean(twelve_pp_improvement):0.2f}%")
print(f"Percentage of cases where we observe 16 percentage point improvement from random noise: {100.*np.mean(sixteen_pp_improvement):0.2f}%")
print(f"Percentage of cases where we observe 20 percentage point improvement from random noise: {100.*np.mean(twenty_pp_improvement):0.2f}%")

Percentage of cases where we observe 8 percentage point improvement from random noise: 33.77%
Percentage of cases where we observe 12 percentage point improvement from random noise: 23.83%
Percentage of cases where we observe 16 percentage point improvement from random noise: 16.02%
Percentage of cases where we observe 20 percentage point improvement from random noise: 9.89%


### Completing the Idea: Comparison Without Software Bugs

Here is the key context for applying the thought experiment to practical evaluation: when we make changes to policy architectures, training datasets, hyperparameters, etc, we __might not be improving the overall performance__. The systems we are engaging with are profoundly complex. As such, the sum of our innovations in a particular phase of a project may be the tragic tale of difficult research: a sound and fury, signifying nothing. 

As such, even when we avoid the bug described in the preceding section, we must still imagine that the new policy, while qualitatively different than the baseline, may still not have improved in average performance. We might be exactly in the realm of the preceding section! 

### Empirical Performance Gaps are Not Enough at Small Sample Sizes
The complexity of small-sample distributions, even for low-dimensional evaluation measures, demands a more principled approach to determining _significant_ (i.e., repeatable and/or generalizable) improvements in policy performance. 

Enter NSCORE. Let's consider the same evaluation 'bug' as before using the NSCORE SAVI framework. 

In [11]:
alpha = 0.05

nscore_test = BernoulliNsmTest(alternative=Hypothesis.P0LessThanP1, alpha=alpha, c=np.arange(2)/1.)

In [12]:
n_fictitious_evaluation_sequences = 1000
n_evaluations_per_sequence = 25 

p0 = 0.57       # Baseline succeeds 57% of the time
p1 = 0.57       # Code bug means we were loading the baseline here as well! So it is also 57%

# Count the times we get a false positive. 
nscore_falsely_decides = np.zeros(n_fictitious_evaluation_sequences)

np.random.seed(42)

for i in range(n_fictitious_evaluation_sequences):
    data_baseline = np.random.binomial(1, p0, n_evaluations_per_sequence)
    data_new_policy = np.random.binomial(1, p1, n_evaluations_per_sequence)

    nscore_test.reset()
    result = nscore_test.run_on_sequence(data_baseline, data_new_policy)

    if result.decision == Decision.AcceptAlternative:
        nscore_falsely_decides[i] += 1.0 


print(f"Percentage of cases where NSCORE falsely decides for the alternative: {100.*np.mean(nscore_falsely_decides):0.2f}% (should be less than {100.*alpha}%)")

Percentage of cases where NSCORE falsely decides for the alternative: 1.20% (should be less than 5.0)


### Rigorous Evaluation Decisions are Important
As just illustrated, using changes to empirical performance alone as justification for policy improvement can give highly distorted and/or poorly calibrated notions of confidence. Were we to use 20% improvement as a benchmark, for example, our true confidence in improvement would only be ~90%. NSCORE (and other rigorous testing methods) formalize this level of confidence, turning it into a common evidential currency.  

## Saving Practitioner Effort: Sample-Efficient _Sequential_ Evaluation with NSCORE
The preceding examples do not reflect the true difficulty of the comparison problem, because they fixed in advance the true, _known_ parameters of the evaluation scores. In reality, we only know that $\mu_0 \in [0, 1]$ and $\mu_1 \in [0, 1]$. This means that tests can be very easy, very hard, or somewhere in between. This is why we need a sequential procedure. Let's see some examples:

In [13]:
# EASY SETTING
n_fictitious_evaluation_sequences = 1000
n_evaluations_per_sequence = 1000

p0 = 0.10       # Baseline succeeds 57% of the time
p1 = 0.90       # Code bug means we were loading the baseline here as well! So it is also 57%

# Count the times we get a false positive. 
nscore_easy_time_to_decision = np.zeros(n_fictitious_evaluation_sequences)
nscore_easy_correct_decisions = np.zeros(n_fictitious_evaluation_sequences)

np.random.seed(31415)

for i in range(n_fictitious_evaluation_sequences):
    data_baseline = np.random.binomial(1, p0, n_evaluations_per_sequence)
    data_new_policy = np.random.binomial(1, p1, n_evaluations_per_sequence)

    nscore_test.reset()
    result = nscore_test.run_on_sequence(data_baseline, data_new_policy)

    if result.decision == Decision.AcceptAlternative:
        nscore_easy_time_to_decision[i] = result.info["Time"]
        nscore_easy_correct_decisions[i] = 1.0


print(f"Percentage of cases where NSCORE correctly decides for the alternative: {100.*np.mean(nscore_easy_correct_decisions):0.2f}%")
print(f"Average time-to-decision for NSCORE in this EASY setting: {np.mean(nscore_easy_time_to_decision):0.2f}")

Percentage of cases where NSCORE correctly decides for the alternative: 100.00%
Average time-to-decision for NSCORE in this EASY setting: 7.24


In [14]:
# MEDIUM SETTING
n_fictitious_evaluation_sequences = 1000
n_evaluations_per_sequence = 1000

p0 = 0.35       # Baseline succeeds 57% of the time
p1 = 0.65       # Code bug means we were loading the baseline here as well! So it is also 57%

# Count the times we get a false positive. 
nscore_medium_time_to_decision = np.zeros(n_fictitious_evaluation_sequences)
nscore_medium_correct_decisions = np.zeros(n_fictitious_evaluation_sequences)

np.random.seed(31415)

for i in range(n_fictitious_evaluation_sequences):
    data_baseline = np.random.binomial(1, p0, n_evaluations_per_sequence)
    data_new_policy = np.random.binomial(1, p1, n_evaluations_per_sequence)

    nscore_test.reset()
    result = nscore_test.run_on_sequence(data_baseline, data_new_policy)

    if result.decision == Decision.AcceptAlternative:
        nscore_medium_time_to_decision[i] = result.info["Time"]
        nscore_medium_correct_decisions[i] = 1.0
    else:
        nscore_medium_time_to_decision[i] = n_evaluations_per_sequence


print(f"Percentage of cases where NSCORE correctly decides for the alternative: {100.*np.mean(nscore_medium_correct_decisions):0.2f}%")
print(f"Average time-to-decision for NSCORE in this MEDIUM setting: {np.mean(nscore_medium_time_to_decision):0.2f}")

Percentage of cases where NSCORE correctly decides for the alternative: 100.00%
Average time-to-decision for NSCORE in this MEDIUM setting: 43.10


In [15]:
# HARD SETTING
n_fictitious_evaluation_sequences = 1000
n_evaluations_per_sequence = 1000

p0 = 0.75       # Baseline succeeds 57% of the time
p1 = 0.85       # Code bug means we were loading the baseline here as well! So it is also 57%

# Count the times we get a false positive. 
nscore_hard_time_to_decision = np.zeros(n_fictitious_evaluation_sequences)
nscore_hard_correct_decisions = np.zeros(n_fictitious_evaluation_sequences)

np.random.seed(31415)

for i in tqdm(range(n_fictitious_evaluation_sequences)):
    data_baseline = np.random.binomial(1, p0, n_evaluations_per_sequence)
    data_new_policy = np.random.binomial(1, p1, n_evaluations_per_sequence)

    nscore_test.reset()
    result = nscore_test.run_on_sequence(data_baseline, data_new_policy)

    if result.decision == Decision.AcceptAlternative:
        nscore_hard_time_to_decision[i] = result.info["Time"]
        nscore_hard_correct_decisions[i] = 1.0
    else:
        nscore_hard_time_to_decision[i] = n_evaluations_per_sequence


print(f"Percentage of cases where NSCORE correctly decides for the alternative: {100.*np.mean(nscore_hard_correct_decisions):0.2f}%")
print(f"Average time-to-decision for NSCORE in this HARD setting: {np.mean(nscore_hard_time_to_decision):0.2f}")

Percentage of cases where NSCORE correctly decides for the alternative: 99.40%
Average time-to-decision for NSCORE in this HARD setting: 275.74


### We Don't Know How Many Samples We Need in Advance!
This is an impossible problem for the robot evaluator, if they have to try and specify a batch size (because there isn't one!). And, we __cannot__ simply apply a batch procedure multiple times (this is, again, p-hacking). 

As an example, imagine the evaluator sets a nominal batch size of 25, but is willing to go up to 200 total evaluations per policy, and keeps running the associated Barnard test in a repeated fashion every 25 trials. This __invalidates__ statistical guarantees, as shown below:

In [19]:
barnard_test = BarnardExactTest(alternative=Hypothesis.P0LessThanP1, alpha=alpha)

n_fictitious_evaluation_sequences = 1000
n_evaluations_per_sequence = 200
key_idx = np.arange(1, 9) * 25

p0 = 0.7       # Baseline succeeds 70% of the time
p1 = 0.7       # Code bug means we were loading the baseline here as well! So it is also 70%

# Count the times we get a false positive. 
barnard_false_positives = np.zeros(n_fictitious_evaluation_sequences)

np.random.seed(31415)

for i in tqdm(range(n_fictitious_evaluation_sequences)):
    data_baseline = np.random.binomial(1, p0, n_evaluations_per_sequence)
    data_new_policy = np.random.binomial(1, p1, n_evaluations_per_sequence)
    run_not_finished = True 

    for j in range(key_idx.shape[0]):
        if run_not_finished:
            current_idx = int(key_idx[j])
            result = barnard_test.run_on_sequence(data_baseline[:current_idx], data_new_policy[:current_idx])

            if result.decision == Decision.AcceptAlternative:
                barnard_false_positives[i] = 1.0
                run_not_finished = False
        
        else:
            pass

print(f"Percentage of cases where Barnard Test falsely decides for the alternative: {100.*np.mean(barnard_false_positives):0.2f}% (should be greater than {100.*alpha:0.2f}%)")


100%|██████████| 1000/1000 [02:26<00:00,  6.83it/s]

Percentage of cases where Barnard Test falsely decides for the alternative: 13.90% (should be greater than 0.05)


### Sequential Evaluation is Necessary for Sample Efficiency
Batch procedures cannot adapt to varying problem difficulty without invalidating their risk-level control. Therefore, though they are better able to distinguish when the alternative is true, it is less useful because we cannot quantify how likely they are to be wrong! 

## NSCORE: Extending to General Performance Measures
One may have noticed that in each of the preceding instances, we have used Bernoulli data as a simple proxy to demonstrate features of the evaluation problem. However, our previous work [STEP](https://github.com/TRI-ML/sequentialized_barnard_tests) currently gives state-of-the-art, near-optimal testing procedures for Bernoulli data. In the small-$N$ regime of robotic evaluation, the optimal approach is to use STEP for the comparison problems. 

What, then, is the benefit of NSCORE? Well, many performance measures of interest are not only not binary, they don't belong to any parametric family of distributions! Some examples: state and action costs in classical control tasks subject to noisy dynamics; reinforcement learning (RL) rewards; and behavioral measures of robot performance (e.g., jitter measures for dexterous manipulation). This is where NSCORE _complements_ STEP, by allowing the evaluator to also consider highly-complex metrics with minimal loss in sample efficiency. 

### Setting 1: Parametric (Partial Credit) Measures

### Setting 2: Nonparametric (General) Performance Measures
We give the example of comparing standard RL training algorithms on the Mujoco benchmark task suite. 